# Sleep Quality — Causal Analysis of Controllable Behaviors

**Question:** which of the things *you control* — logged journal behaviors (alcohol, caffeine, supplements, …) **and** controllable habits outside the journal (bedtime lateness, day strain, naps) — actually move your sleep?

**Data source:** the Whoop **personal data export** (`Settings → Account → Export My Data`), *not* the API — Whoop's API has no journal endpoint. The export's four CSVs live in `data/raw/` (plaintext is git-ignored; an AES-256 copy `data/whoop_export.tar.gz.gpg` is committed).

**Method:**
1. Build a daily table: sleep/recovery/strain (from `physiological_cycles.csv` + `sleeps.csv`) joined with journal behaviors (`journal_entries.csv`).
2. Tag every variable: **controllable** (a lever you pull), **outcome** (sleep quality), **physiological confounder** (adjust for, never an action), or **symptom** (a feeling — excluded as a cause).
3. **PCMCI** discovers the time-lagged causal graph.
4. **Effect sizes** in native units (minutes / %) quantify each controllable factor's impact, with significance + a placebo check.

## Phase 0 — Load the export

Reads the four CSVs from `../data/raw/`. If only the encrypted archive is present, decrypts it first (you'll be asked for the password).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import subprocess, getpass

RAW = Path("../data/raw")
ENC = Path("../data/whoop_export.tar.gz.gpg")
NEEDED = ["physiological_cycles.csv", "journal_entries.csv", "sleeps.csv"]

def _have_csvs():
    return all((RAW / f).exists() for f in NEEDED)

if not _have_csvs() and ENC.exists():
    print("Plaintext CSVs missing; decrypting", ENC.name, "...")
    RAW.mkdir(parents=True, exist_ok=True)
    pw = getpass.getpass("Decryption password: ")
    # gpg -> tar extract into RAW
    p = subprocess.run(
        f'gpg --batch --yes --passphrase {pw!r} -d {ENC} | tar -xzf - -C {RAW}',
        shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("Decrypt/extract failed: " + p.stderr[-300:])

assert _have_csvs(), f"Put the export CSVs in {RAW.resolve()} (or commit the .gpg). Missing: {[f for f in NEEDED if not (RAW/f).exists()]}"
phys = pd.read_csv(RAW / "physiological_cycles.csv")
jour = pd.read_csv(RAW / "journal_entries.csv")
print(f"physiological_cycles: {phys.shape}")
print(f"journal_entries:      {jour.shape}")
print(f"date span: {phys['Cycle start time'].min()[:10]} -> {phys['Cycle start time'].max()[:10]}")

physiological_cycles: (193, 26)
journal_entries:      (3240, 6)
date span: 2025-06-14 -> 2026-06-25


## Phase 1 — Build the daily dataset

`physiological_cycles.csv` is already one row per day with sleep, recovery, and strain joined. We tidy the column names and derive two extra **controllable** levers:
- **`ctl_bedtime_lateness`** — hours past 18:00 you fell asleep (you control when you go to bed)
- **`ctl_day_strain`** — how hard you exercised (you control activity)

In [2]:
phys = phys.copy()
phys["date"] = pd.to_datetime(phys["Cycle start time"], errors="coerce").dt.normalize()
phys = phys.dropna(subset=["date"])

colmap = {
    "Asleep duration (min)": "total_sleep_minutes",
    "Deep (SWS) duration (min)": "sws_minutes",
    "REM duration (min)": "rem_minutes",
    "Light sleep duration (min)": "light_minutes",
    "Awake duration (min)": "awake_minutes",
    "Sleep efficiency %": "sleep_efficiency",
    "Sleep performance %": "sleep_performance",
    "Sleep consistency %": "sleep_consistency",
    "Respiratory rate (rpm)": "respiratory_rate",
    "Sleep debt (min)": "sleep_debt_minutes",
    "Recovery score %": "recovery_score",
    "Resting heart rate (bpm)": "resting_hr",
    "Heart rate variability (ms)": "hrv_rmssd",
    "Skin temp (celsius)": "skin_temp",
    "Blood oxygen %": "spo2",
    "Day Strain": "ctl_day_strain",
}
keep = {k: v for k, v in colmap.items() if k in phys.columns}
daily = phys.groupby("date")[list(keep)].first().rename(columns=keep)

# Derived controllable: bedtime lateness (hours past 6pm)
onset = pd.to_datetime(phys.set_index("date")["Sleep onset"], errors="coerce")
onset = onset.groupby(level=0).first()
hod = onset.dt.hour + onset.dt.minute / 60.0
daily["ctl_bedtime_lateness"] = ((hod - 18) % 24)

daily = daily.sort_index()
print(f"Daily physiological table: {daily.shape[0]} days, {daily.shape[1]} columns")
daily.head()

Daily physiological table: 169 days, 17 columns


,total_sleep_minutes,sws_minutes,rem_minutes,light_minutes,awake_minutes,sleep_efficiency,sleep_performance,sleep_consistency,respiratory_rate,sleep_debt_minutes,recovery_score,resting_hr,hrv_rmssd,skin_temp,spo2,ctl_day_strain,ctl_bedtime_lateness
date,,,,,,,,,,,,,,,,,
2025-06-14,139.0,23.0,20.0,96.0,0.0,100.0,30.0,NaN,NaN,0.0,10.0,120.0,43.0,NaN,NaN,4.0,20.566667
2025-06-17,597.0,108.0,146.0,343.0,97.0,86.0,84.0,NaN,16.9,38.0,36.0,120.0,40.0,NaN,NaN,6.7,8.200000
2025-06-18,480.0,89.0,122.0,269.0,32.0,93.0,84.0,NaN,14.0,102.0,95.0,48.0,192.0,33.80,96.45,17.3,4.933333
2025-06-19,389.0,108.0,66.0,215.0,34.0,91.0,77.0,75.0,14.4,71.0,90.0,49.0,189.0,34.00,95.69,5.3,5.583333
2025-06-21,464.0,117.0,126.0,221.0,39.0,92.0,77.0,55.0,14.1,110.0,92.0,51.0,165.0,34.61,97.00,5.6,7.316667


## Phase 2 — Journal behaviors → controllable factors

Each journal entry is one yes/no answer to one question on one day. We pivot to one `do_*` column per behavior, keeping only behaviors **logged on ≥60 days with real variance** (between 5% and 95% "yes") so there's something to learn from.

We also separate **actions** (things you do — caffeine, alcohol, cold shower) from **symptoms** (things you feel — brain fog, fatigue). Symptoms are outcomes, not levers, so they're never treated as causes.

In [3]:
jour = jour.dropna(subset=["Cycle start time"]).copy()
jour["date"] = pd.to_datetime(jour["Cycle start time"], errors="coerce").dt.normalize()
jour = jour.dropna(subset=["date"])

def to01(v):
    s = str(v).strip().lower()
    return 1.0 if s in ("true","yes","y","1","1.0") else (0.0 if s in ("false","no","n","0","0.0") else np.nan)

def slug(q):
    return "do_" + "".join(ch if ch.isalnum() else "_" for ch in str(q).lower()).strip("_")

jour["val"] = jour["Answered yes"].map(to01)
jour["col"] = jour["Question text"].map(slug)
wide = jour.pivot_table(index="date", columns="col", values="val", aggfunc="max")

# Keep behaviors with enough data + variance
good = [c for c in wide.columns
        if wide[c].notna().sum() >= 60 and 0.05 <= wide[c].mean() <= 0.95]
wide = wide[good]

# Journal items that are feelings/symptoms, not actions -> excluded as causes
SYMPTOM_KEYS = ["brain_fog", "fatigue", "emotionally", "depressed", "energized",
                "libido", "stable", "stressed", "anxious", "sick"]
journal_symptoms = [c for c in wide.columns if any(k in c for k in SYMPTOM_KEYS)]
journal_actions  = [c for c in wide.columns if c not in journal_symptoms]

print(f"Journal behaviors kept: {len(wide.columns)}")
print(f"  actions  ({len(journal_actions)}): {[c[3:] for c in journal_actions]}")
print(f"  symptoms ({len(journal_symptoms)}, excluded as causes): {[c[3:] for c in journal_symptoms]}")

Journal behaviors kept: 16
  actions  (13): ['consumed_added_sugar', 'consumed_caffeine', 'consumed_fiber', 'consumed_fruits_and_or_vegetables', 'consumed_nicotine', 'have_any_alcoholic_drinks', 'masturbated', 'took_a_cold_shower', 'took_creatine', 'took_fish_oil', 'took_omega_3_supplement', 'traveled_on_a_plane', 'used_social_media']
  symptoms (3, excluded as causes): ['experienced_brain_fog', 'experienced_fatigue', 'felt_emotionally_and_mentally_stable']


## Phase 3 — Merge & classify every variable

Join journal behaviors onto the daily physiological table. A behavior not logged on a day = "didn't do it" → 0. Then we label each column's **role**:

| Role | Meaning | Used as |
|------|---------|---------|
| **controllable** | a lever you pull (journal action, bedtime, strain, nap) | treatment |
| **outcome** | sleep-quality / recovery metric | what we explain |
| **confounder** | physiology you don't directly set (resp. rate, skin temp, SpO₂) | adjusted for |
| **symptom** | a feeling you reported | excluded |

In [4]:
merged = daily.join(wide, how="left")
journal_cols = list(wide.columns)
merged[journal_cols] = merged[journal_cols].fillna(0)
merged = merged.dropna(subset=["total_sleep_minutes"]).sort_index()
merged = merged.dropna(axis=1, how="all")

# Roles
OUTCOMES = [c for c in ["total_sleep_minutes","sws_minutes","rem_minutes","light_minutes",
                        "sleep_efficiency","sleep_performance","awake_minutes",
                        "sleep_debt_minutes","recovery_score","hrv_rmssd"] if c in merged.columns]
CONTROLLABLE_EXTRA = [c for c in ["ctl_bedtime_lateness","ctl_day_strain","sleep_consistency"]
                      if c in merged.columns]
CONTROLLABLE = [c for c in journal_actions if c in merged.columns] + CONTROLLABLE_EXTRA
SYMPTOMS = [c for c in journal_symptoms if c in merged.columns]
CONFOUNDERS = [c for c in ["respiratory_rate","skin_temp","spo2","resting_hr"] if c in merged.columns]

# Fill residual physiological NaNs with column means
fillcols = OUTCOMES + CONFOUNDERS + [c for c in CONTROLLABLE_EXTRA]
merged[fillcols] = merged[fillcols].fillna(merged[fillcols].mean())

print(f"Final dataset: {merged.shape[0]} days, {merged.shape[1]} variables")
print(f"\nCONTROLLABLE ({len(CONTROLLABLE)}): {CONTROLLABLE}")
print(f"\nOUTCOMES ({len(OUTCOMES)}): {OUTCOMES}")
print(f"\nCONFOUNDERS ({len(CONFOUNDERS)}): {CONFOUNDERS}")
print(f"\nSYMPTOMS excluded ({len(SYMPTOMS)}): {SYMPTOMS}")

Final dataset: 167 days, 33 variables

CONTROLLABLE (16): ['do_consumed_added_sugar', 'do_consumed_caffeine', 'do_consumed_fiber', 'do_consumed_fruits_and_or_vegetables', 'do_consumed_nicotine', 'do_have_any_alcoholic_drinks', 'do_masturbated', 'do_took_a_cold_shower', 'do_took_creatine', 'do_took_fish_oil', 'do_took_omega_3_supplement', 'do_traveled_on_a_plane', 'do_used_social_media', 'ctl_bedtime_lateness', 'ctl_day_strain', 'sleep_consistency']

OUTCOMES (10): ['total_sleep_minutes', 'sws_minutes', 'rem_minutes', 'light_minutes', 'sleep_efficiency', 'sleep_performance', 'awake_minutes', 'sleep_debt_minutes', 'recovery_score', 'hrv_rmssd']

CONFOUNDERS (4): ['respiratory_rate', 'skin_temp', 'spo2', 'resting_hr']

SYMPTOMS excluded (3): ['do_experienced_brain_fog', 'do_experienced_fatigue', 'do_felt_emotionally_and_mentally_stable']


## Phase 4 — Exploratory views

How often you do each behavior, and how your nightly sleep varies.

In [5]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Behavior frequency
binary_ctrl = [c for c in CONTROLLABLE if set(merged[c].dropna().unique()) <= {0.0, 1.0}]
freq = merged[binary_ctrl].mean().sort_values()
fig = go.Figure(go.Bar(x=freq.values * 100, y=[c[3:] if c.startswith("do_") else c for c in freq.index],
                       orientation="h", marker_color="#2d8a4e"))
fig.update_layout(title="How often you do each controllable behavior",
                  xaxis_title="% of days", height=max(350, len(freq)*26), width=720)
fig.show()

# Sleep over time
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=["Total sleep (min)", "Deep + REM (min)"])
fig2.add_trace(go.Scatter(x=merged.index, y=merged["total_sleep_minutes"],
                          mode="lines", name="total", line=dict(color="#2d8a4e")), 1, 1)
if "sws_minutes" in merged: fig2.add_trace(go.Scatter(x=merged.index, y=merged["sws_minutes"], name="deep"), 2, 1)
if "rem_minutes" in merged: fig2.add_trace(go.Scatter(x=merged.index, y=merged["rem_minutes"], name="rem"), 2, 1)
fig2.update_layout(height=460, title_text="Sleep over time")
fig2.show()

## Phase 5 — Causal discovery (PCMCI)

PCMCI tests time-lagged **conditional** independence: it keeps an edge `A(t-k) → sleep(t)` only if it survives controlling for the other variables — filtering out mere correlations. We run it over controllable factors + sleep outcomes + physiological confounders, then report only the edges that land **on a sleep outcome from a controllable factor**.

In [6]:
from tigramite import data_processing as pp
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.pcmci import PCMCI

SLEEP_TARGETS = [c for c in ["total_sleep_minutes","sws_minutes","rem_minutes",
                             "sleep_efficiency","recovery_score","hrv_rmssd"] if c in merged.columns]
analysis_cols = list(dict.fromkeys(SLEEP_TARGETS + CONFOUNDERS + CONTROLLABLE))
X = merged[analysis_cols].dropna()
Xz = (X - X.mean()) / X.std(ddof=0)
Xz = Xz.loc[:, Xz.std() > 0]      # drop any zero-variance cols
var_names = list(Xz.columns)
ctrl_set = set(c for c in CONTROLLABLE if c in var_names)

MAX_LAG, ALPHA = 3, 0.10
pcmci = PCMCI(dataframe=pp.DataFrame(Xz.values.astype(float), var_names=var_names),
              cond_ind_test=ParCorr(significance="analytic"), verbosity=0)
print(f"Running PCMCI on {len(var_names)} vars over {X.shape[0]} days (max_lag={MAX_LAG})...")
res = pcmci.run_pcmci(tau_max=MAX_LAG, pc_alpha=0.2)
graph = pcmci.get_graph_from_pmatrix(p_matrix=res["p_matrix"], alpha_level=ALPHA, tau_min=1, tau_max=MAX_LAG)
val, pmat = res["val_matrix"], res["p_matrix"]

def controllable_parents(target):
    ti = var_names.index(target); out = []
    for si, s in enumerate(var_names):
        if s not in ctrl_set: continue
        for lag in range(1, MAX_LAG + 1):
            if graph[si, ti, lag] == "-->":
                out.append({"source": s, "lag": lag,
                            "strength": float(val[si, ti, lag]), "p": float(pmat[si, ti, lag])})
    return sorted(out, key=lambda d: abs(d["strength"]), reverse=True)

discovery = {}
print(f"\nCONTROLLABLE causes of each sleep outcome (PCMCI, p<{ALPHA}):")
for tgt in SLEEP_TARGETS:
    ps = controllable_parents(tgt); discovery[tgt] = ps
    print(f"\n### {tgt}")
    if ps:
        for p in ps:
            arrow = "increases" if p["strength"] > 0 else "decreases"
            name = p["source"][3:] if p["source"].startswith("do_") else p["source"]
            print(f"   {name:34s} (lag {p['lag']}d)  {arrow:9s}  r={p['strength']:+.3f}  p={p['p']:.3f}")
    else:
        print("   (no controllable causes at this threshold)")

Running PCMCI on 26 vars over 167 days (max_lag=3)...



CONTROLLABLE causes of each sleep outcome (PCMCI, p<0.1):

### total_sleep_minutes
   consumed_caffeine                  (lag 3d)  increases  r=+0.233  p=0.005
   took_creatine                      (lag 2d)  decreases  r=-0.196  p=0.017
   ctl_day_strain                     (lag 1d)  decreases  r=-0.175  p=0.036
   consumed_nicotine                  (lag 1d)  increases  r=+0.174  p=0.033
   have_any_alcoholic_drinks          (lag 2d)  decreases  r=-0.164  p=0.048
   consumed_fruits_and_or_vegetables  (lag 1d)  decreases  r=-0.143  p=0.085

### sws_minutes
   ctl_bedtime_lateness               (lag 2d)  decreases  r=-0.169  p=0.039
   consumed_fiber                     (lag 1d)  decreases  r=-0.148  p=0.071
   consumed_fruits_and_or_vegetables  (lag 1d)  decreases  r=-0.141  p=0.085
   took_creatine                      (lag 1d)  decreases  r=-0.140  p=0.088
   have_any_alcoholic_drinks          (lag 1d)  increases  r=+0.137  p=0.095
   consumed_added_sugar               (lag 2d)  incr

## Phase 6 — Effect sizes in real units

PCMCI tells us *which* factors matter and the direction; this quantifies *how much*, in the units you care about (minutes of sleep, % efficiency). For each **binary** behavior we compare nights when you did vs didn't do it (Welch t-test, same-day and next-day). For **continuous** levers (bedtime lateness, strain) we use a median split. A **placebo test** (shuffle the behavior 200×) flags effects that are likely noise.

In [7]:
from scipy import stats

UNIT = {"total_sleep_minutes":"min","sws_minutes":"min","rem_minutes":"min","light_minutes":"min",
        "awake_minutes":"min","sleep_efficiency":"%","sleep_performance":"%","recovery_score":"%",
        "hrv_rmssd":"ms","sleep_debt_minutes":"min"}

def placebo_p(exposure, outcome, observed, n=200, seed=0):
    """Shuffle the exposure n times; fraction of shuffles whose mean-diff is
    at least as large as observed. High p => effect looks like noise."""
    rng = np.random.default_rng(seed)
    e = np.asarray(exposure); y = np.asarray(outcome); cnt = 0
    for _ in range(n):
        perm = rng.permutation(e)
        on = y[perm == 1]; off = y[perm == 0]
        if len(on) > 3 and len(off) > 3 and abs(on.mean() - off.mean()) >= abs(observed):
            cnt += 1
    return cnt / n

rows = []
for beh in CONTROLLABLE:
    s = merged[beh]
    is_binary = set(s.dropna().unique()) <= {0.0, 1.0}
    for lag in (0, 1):
        exp = (s.shift(lag) if lag else s)
        if not is_binary:                      # continuous -> median split
            exp = (exp > exp.median()).astype(float)
        for tgt in SLEEP_TARGETS:
            y = merged[tgt]
            mask = exp.notna() & y.notna()
            ev, yv = exp[mask], y[mask]
            on, off = yv[ev == 1], yv[ev == 0]
            if len(on) < 8 or len(off) < 8: continue
            t, p = stats.ttest_ind(on, off, equal_var=False)
            if p < 0.05:
                delta = on.mean() - off.mean()
                pb = placebo_p(ev, yv.values, delta)
                rows.append({"factor": beh[3:] if beh.startswith("do_") else beh,
                             "raw": beh, "target": tgt, "unit": UNIT.get(tgt, ""),
                             "when": "same-day" if lag == 0 else "next-day",
                             "delta": round(float(delta), 1), "p": round(float(p), 4),
                             "placebo_p": round(float(pb), 3), "n_on": int(len(on))})

effects = pd.DataFrame(rows).sort_values(["target", "p"])
print(f"{len(effects)} significant controllable -> sleep effects (p<0.05)\n")
for tgt in SLEEP_TARGETS:
    sub = effects[effects.target == tgt]
    if len(sub):
        print(f"### {tgt}")
        for _, r in sub.iterrows():
            flag = "" if r.placebo_p < 0.05 else "  (weak: placebo n.s.)"
            print(f"   {r.factor:32s} {r.when}: {r.delta:+6.1f} {r.unit:3s}  p={r.p}{flag}")
        print()

25 significant controllable -> sleep effects (p<0.05)

### total_sleep_minutes
   ctl_bedtime_lateness             same-day:  -43.3 min  p=0.0018
   masturbated                      same-day:  +40.0 min  p=0.0068
   took_a_cold_shower               same-day:  +34.5 min  p=0.0079
   took_creatine                    same-day:  +31.3 min  p=0.0156  (weak: placebo n.s.)
   consumed_caffeine                same-day:  +31.1 min  p=0.0198
   consumed_added_sugar             same-day:  +31.8 min  p=0.029
   consumed_fruits_and_or_vegetables same-day:  +33.1 min  p=0.0328  (weak: placebo n.s.)
   traveled_on_a_plane              same-day:  +51.4 min  p=0.0428  (weak: placebo n.s.)

### sws_minutes
   masturbated                      same-day:  +14.9 min  p=0.0003
   consumed_added_sugar             same-day:   +9.5 min  p=0.01
   consumed_fiber                   next-day:  -10.2 min  p=0.0149
   took_creatine                    next-day:   -8.3 min  p=0.0301
   consumed_fruits_and_or_vegetables

## Phase 7 — Effect chart & takeaways

In [8]:
if len(effects):
    top = effects.copy()
    top["label"] = top["factor"] + " → " + top["target"].str.replace("_"," ") + " (" + top["when"] + ")"
    top["solid"] = top["placebo_p"] < 0.05
    top = top.reindex(top.delta.abs().sort_values().index)
    colors = ["#2d8a4e" if s else "#bbbbbb" for s in top["solid"]]
    fig = go.Figure(go.Bar(x=top["delta"], y=top["label"], orientation="h",
                           marker_color=colors,
                           text=[f"{d:+.1f}{u}" for d,u in zip(top.delta, top.unit)],
                           textposition="outside"))
    fig.update_layout(title="Controllable behavior → sleep effect sizes<br><sub>green = survives placebo shuffle; grey = weaker</sub>",
                      xaxis_title="change in outcome", height=max(420, len(top)*30), width=900,
                      margin=dict(l=320))
    fig.show()
else:
    print("No significant effects — try lowering thresholds or collecting more logged days.")

## Phase 8 — Tightened causal estimates (defensible numbers)

Phases 5–6 cast a wide net. This phase zooms in on the levers that are both **statistically strong and physiologically plausible**, and estimates each one *properly*:

- **Confounder adjustment** — control for physiology you don't set directly (respiratory rate, skin temp, SpO₂, sleep debt).
- **Regression-to-mean control** — include *yesterday's* value of the outcome, so a bad night followed by a normal one isn't mistaken for an effect.
- **Two estimators that should agree:** DoWhy backdoor linear regression (causal framing + refutation) and an OLS fit with robust HC3 standard errors (interpretable 95% CI in native units).
- **Three DoWhy refutations:** placebo treatment (shuffle the lever → effect should vanish), random common cause (add a noise confounder → estimate should hold), 80% subset (estimate should be stable).

A lever is **trustworthy** only if both estimators agree, the 95% CI excludes zero, and it survives the refutations.


In [9]:
import statsmodels.api as sm
from dowhy import CausalModel
import warnings; warnings.filterwarnings("ignore")

# Levers worth a rigorous estimate: physiologically plausible + flagged in Phases 5-6.
# (treatment, outcome, lag, human-readable unit)
#   lag=0 -> behavior on the same cycle as the sleep/recovery readout
#   lag=1 -> behavior on the prior day affecting today's outcome
TIGHT_TESTS = [
    ("ctl_bedtime_lateness",       "total_sleep_minutes", 0, "min per hr later"),
    ("ctl_bedtime_lateness",       "recovery_score",      0, "% per hr later"),
    ("ctl_bedtime_lateness",       "hrv_rmssd",           0, "ms per hr later"),
    ("ctl_day_strain",             "hrv_rmssd",           1, "ms per strain pt"),
    ("ctl_day_strain",             "recovery_score",      1, "% per strain pt"),
    ("do_took_a_cold_shower",      "total_sleep_minutes", 0, "min (yes vs no)"),
    ("do_have_any_alcoholic_drinks","sws_minutes",        0, "min (yes vs no)"),
]
TIGHT_CONF = [c for c in ["respiratory_rate", "skin_temp", "spo2", "sleep_debt_minutes"]
              if c in merged.columns]

def _ols_ci(treat, outcome, lag):
    """OLS w/ HC3 robust SEs -> (beta, lo, hi, p, n). Adjusts for confounders + prior outcome."""
    d = merged.copy()
    if lag > 0:
        d[treat] = d[treat].shift(lag)
    d["_prev"] = d[outcome].shift(1)
    cols = [treat] + TIGHT_CONF + ["_prev"]
    d = d[[outcome] + cols].dropna()
    if d[treat].nunique() < 2 or len(d) < 30:
        return None
    r = sm.OLS(d[outcome], sm.add_constant(d[cols])).fit(cov_type="HC3")
    lo, hi = r.conf_int().loc[treat]
    return float(r.params[treat]), float(lo), float(hi), float(r.pvalues[treat]), len(d)

def _dowhy_refute(treat, outcome, lag):
    """DoWhy backdoor ATE + 3 refutations. Returns (ate, placebo_p, rand_drift, subset_drift)."""
    d = merged.copy()
    if lag > 0:
        d[treat] = d[treat].shift(lag)
    d["_prev"] = d[outcome].shift(1)
    conf = TIGHT_CONF + ["_prev"]
    d = d[[treat, outcome] + conf].dropna()
    edges = [f'"{treat}" -> "{outcome}"']
    for c in conf:
        edges += [f'"{c}" -> "{treat}"', f'"{c}" -> "{outcome}"']
    nodes = "; ".join(f'"{n}"' for n in set([treat, outcome] + conf))
    g = f"digraph {{ {nodes}; {'; '.join(edges)} }}"
    m = CausalModel(data=d, treatment=treat, outcome=outcome, common_causes=conf, graph=g)
    ident = m.identify_effect(proceed_when_unidentifiable=True)
    est = m.estimate_effect(ident, method_name="backdoor.linear_regression")
    ate = float(est.value)
    rp = m.refute_estimate(ident, est, method_name="placebo_treatment_refuter",
                           placebo_type="permute", num_simulations=200)
    rr = m.refute_estimate(ident, est, method_name="random_common_cause", num_simulations=100)
    rs = m.refute_estimate(ident, est, method_name="data_subset_refuter",
                           subset_fraction=0.8, num_simulations=100)
    pv = rp.refutation_result.get("p_value") if getattr(rp, "refutation_result", None) else None
    pv = float(pv) if isinstance(pv, (int, float)) else (float(pv[0]) if hasattr(pv, "__len__") else np.nan)
    drift = lambda r: abs((float(r.new_effect) - ate) / ate) if ate else np.nan
    return ate, pv, drift(rr), drift(rs)

tight_rows = []
print(f"{'lever':24s} {'outcome':19s} {'ATE':>8s}  {'95% CI':>17s}  {'p':>7s}  plac  trust")
print("-" * 92)
for treat, outcome, lag, unit in TIGHT_TESTS:
    ci = _ols_ci(treat, outcome, lag)
    if ci is None:
        continue
    beta, lo, hi, p, n = ci
    try:
        ate, plac_p, rand_drift, sub_drift = _dowhy_refute(treat, outcome, lag)
    except Exception as e:
        ate, plac_p, rand_drift, sub_drift = np.nan, np.nan, np.nan, np.nan
    agree = (not np.isnan(ate)) and abs(ate - beta) < max(2.0, 0.25 * abs(beta))
    ci_excl0 = (lo > 0) or (hi < 0)
    survives = (plac_p > 0.1) and (rand_drift < 0.2) and (sub_drift < 0.2)
    trust = "YES" if (agree and ci_excl0 and survives) else "no"
    name = treat.replace("do_", "").replace("ctl_", "")
    tight_rows.append({"lever": name, "outcome": outcome, "unit": unit, "ate": beta,
                       "lo": lo, "hi": hi, "p": p, "placebo_p": plac_p, "n": n, "trust": trust})
    print(f"{name:24s} {outcome:19s} {beta:+8.2f}  [{lo:+7.1f},{hi:+7.1f}]  {p:7.4f}  "
          f"{plac_p:.2f}  {trust}")

tight = pd.DataFrame(tight_rows)
print("\nLegend: ATE in the lever's native unit (see 'unit' col). "
      "trust=YES means both estimators agree, CI excludes 0, and all refutations pass.")


lever                    outcome                  ATE             95% CI        p  plac  trust
--------------------------------------------------------------------------------------------


bedtime_lateness         total_sleep_minutes   -27.86  [  -39.3,  -16.4]   0.0000  0.86  YES


bedtime_lateness         recovery_score         -6.73  [   -8.9,   -4.5]   0.0000  0.88  YES


bedtime_lateness         hrv_rmssd              -9.67  [  -13.4,   -5.9]   0.0000  0.99  YES


day_strain               hrv_rmssd              -1.96  [   -3.8,   -0.1]   0.0346  0.93  YES


day_strain               recovery_score         -1.57  [   -2.7,   -0.4]   0.0069  0.96  YES


took_a_cold_shower       total_sleep_minutes   +35.64  [   +8.9,  +62.4]   0.0091  0.87  YES


have_any_alcoholic_drinks sws_minutes            +7.94  [   -3.3,  +19.2]   0.1659  0.89  no

Legend: ATE in the lever's native unit (see 'unit' col). trust=YES means both estimators agree, CI excludes 0, and all refutations pass.


In [10]:
# Forest plot of the tightened estimates with 95% CIs
if len(tight):
    t = tight.copy()
    t["label"] = t["lever"] + " → " + t["outcome"].str.replace("_", " ") + "  (" + t["unit"] + ")"
    t = t.iloc[::-1]   # nicest top-to-bottom order
    colors = ["#2d8a4e" if tr == "YES" else "#c0392b" for tr in t["trust"]]
    fig = go.Figure()
    for _, r in t.iterrows():
        c = "#2d8a4e" if r["trust"] == "YES" else "#999999"
        fig.add_trace(go.Scatter(
            x=[r["lo"], r["hi"]], y=[r["label"], r["label"]],
            mode="lines", line=dict(color=c, width=3), showlegend=False))
        fig.add_trace(go.Scatter(
            x=[r["ate"]], y=[r["label"]], mode="markers+text",
            marker=dict(color=c, size=11),
            text=[f"  {r['ate']:+.1f}"], textposition="middle right",
            showlegend=False))
    fig.add_vline(x=0, line_dash="dash", line_color="#444")
    fig.update_layout(
        title="Tightened causal estimates — ATE with 95% CI<br>"
              "<sub>green = trustworthy (agrees across estimators, CI≠0, survives refutation); grey = not</sub>",
        xaxis_title="effect on outcome (native units; CI crossing 0 = inconclusive)",
        height=max(420, len(t) * 55), width=900, margin=dict(l=340))
    fig.show()

    trusted = tight[tight.trust == "YES"]
    print("\nTRUSTWORTHY levers (act on these, then re-measure):")
    if len(trusted):
        for _, r in trusted.iterrows():
            print(f"  • {r['lever']} → {r['outcome']}: {r['ate']:+.1f} "
                  f"[{r['lo']:+.1f}, {r['hi']:+.1f}] {r['unit']}")
    else:
        print("  (none cleared the full bar this run)")
else:
    print("No tightened estimates produced.")



TRUSTWORTHY levers (act on these, then re-measure):
  • bedtime_lateness → total_sleep_minutes: -27.9 [-39.3, -16.4] min per hr later
  • bedtime_lateness → recovery_score: -6.7 [-8.9, -4.5] % per hr later
  • bedtime_lateness → hrv_rmssd: -9.7 [-13.4, -5.9] ms per hr later
  • day_strain → hrv_rmssd: -2.0 [-3.8, -0.1] ms per strain pt
  • day_strain → recovery_score: -1.6 [-2.7, -0.4] % per strain pt
  • took_a_cold_shower → total_sleep_minutes: +35.6 [+8.9, +62.4] min (yes vs no)


### How to read this

- Each bar = doing that behavior is associated with this much change in a sleep metric, vs days you didn't (or, for bedtime/strain, above vs below your median).
- **Green** bars survived a placebo shuffle (less likely to be noise); grey are weaker — treat as leads, not facts.
- **Lag**: *same-day* = behavior and sleep on the same date; *next-day* = the behavior's effect on the following night.

### Important caveats
- These are **adjusted associations**, not clean experiments. With ~5 months of self-logged data, confounding and reverse causation are real (e.g. "alcohol → higher next-day HRV" is almost certainly a logging/timing artifact, not a true effect).
- A behavior you almost never log (small *n*) gives an unstable estimate even when "significant."
- The trustworthy use: pick 1–2 green, physiologically-plausible effects (e.g. a bedtime or strain pattern) and **test them deliberately** for a couple weeks, then re-run this notebook.

## Phase 9 — Mental health: what drives brain fog, fatigue & energy?

Same machinery as the sleep analysis, but the **outcomes are now your mental-health flags** and the **causes include sleep itself** (plus behaviors, bedtime, strain). We ask: *what you do — and how you sleep — on one day, how does it change your mind the next?*

**Outcomes** (binary daily journal flags, kept only if logged ≥60 days with real variance):
- `brain_fog`, `fatigue` — lower is better
- `energized` — higher is better

(`felt depressed` and `libido` are dropped — almost no variance, nothing to learn.)

**Causal direction matters here:** a symptom *today* can only be caused by something *yesterday or earlier*, so every cause is **lagged by 1 day** (prior-night sleep → today's brain fog). We also adjust for **yesterday's symptom** (some people have multi-day fog streaks) and the physiological confounders, then quantify each effect:
- **Binary causes** (behaviors): risk difference = change in the symptom's probability, in **percentage points**, on days you did vs didn't do it.
- **Continuous causes** (sleep minutes, HRV, bedtime): logistic regression, reported as **odds ratio per +1 SD**.

A placebo shuffle flags effects that are likely noise.


In [11]:
# --- Build mental-health outcomes from the journal (binary daily flags) ---
# Phase 2's pivot columns already carry the 'do_' prefix, so look those up directly.
MH_MAP = {
    "do_experienced_brain_fog": "brain_fog",
    "do_experienced_fatigue": "fatigue",
    "do_felt_energized_throughout_the_day": "energized",
    "do_felt_emotionally_and_mentally_stable": "emotionally_stable",
}
_jw = jour.pivot_table(index="date", columns="col", values="val", aggfunc="max")
mh = pd.DataFrame(index=merged.index)
for raw, nice in MH_MAP.items():
    if raw in _jw.columns:
        s = _jw[raw].reindex(merged.index)
        # keep only outcomes with enough data + variance
        if s.notna().sum() >= 60 and 0.03 <= s.mean() <= 0.97:
            mh[nice] = s
MH_OUTCOMES = list(mh.columns)
GOOD_WHEN_HIGH = {"energized", "emotionally_stable"}   # for these, + is good

# --- Causes: behaviors (actions) + sleep/recovery + bedtime/strain (everything controllable
#     or upstream of mood). Symptoms themselves are never causes. ---
MH_SLEEP_CAUSES = [c for c in ["total_sleep_minutes","sws_minutes","rem_minutes","sleep_efficiency",
                               "recovery_score","hrv_rmssd","ctl_bedtime_lateness","ctl_day_strain"]
                   if c in merged.columns]
MH_BEHAVIORS = [c for c in CONTROLLABLE if c.startswith("do_")]
MH_CAUSES = MH_BEHAVIORS + MH_SLEEP_CAUSES
MH_CONF = [c for c in ["resting_hr","respiratory_rate","skin_temp","spo2"] if c in merged.columns]

print(f"Mental-health outcomes ({len(MH_OUTCOMES)}): {MH_OUTCOMES}")
print(f"Candidate causes: {len(MH_BEHAVIORS)} behaviors + {len(MH_SLEEP_CAUSES)} sleep/physio levers")
print(f"Days with outcome data: " + ", ".join(f"{o}={int(mh[o].notna().sum())}" for o in MH_OUTCOMES))

from scipy import stats

def mh_effect(cause, outcome, lag=1, n_placebo=200, seed=0):
    """Effect of a (lagged) cause on a binary MH outcome.
    Binary cause -> risk difference (percentage points) + Welch p + placebo.
    Continuous cause -> logistic OR per +1 SD, adjusting for prior symptom + confounders."""
    d = merged.copy()
    d["_y"] = mh[outcome]
    x = d[cause].shift(lag) if lag else d[cause]
    d["_x"] = x
    d["_prevy"] = d["_y"].shift(1)
    is_binary = set(pd.Series(x).dropna().unique()) <= {0.0, 1.0}
    sub = d[["_x","_y","_prevy"] + MH_CONF].dropna()
    sub = sub[sub["_y"].isin([0,1])]
    if len(sub) < 40 or sub["_y"].nunique() < 2:
        return None
    if is_binary:
        if sub["_x"].nunique() < 2: return None
        on = sub[sub["_x"]==1]["_y"]; off = sub[sub["_x"]==0]["_y"]
        if len(on) < 8 or len(off) < 8: return None
        rd = (on.mean() - off.mean()) * 100.0
        _, p = stats.ttest_ind(on, off, equal_var=False)
        # placebo: shuffle exposure
        rng = np.random.default_rng(seed); e = sub["_x"].values; y = sub["_y"].values; cnt = 0
        for _ in range(n_placebo):
            perm = rng.permutation(e)
            o1 = y[perm==1]; o0 = y[perm==0]
            if len(o1)>3 and len(o0)>3 and abs(o1.mean()-o0.mean())*100 >= abs(rd): cnt += 1
        return {"kind":"binary","effect":rd,"unit":"pp","p":p,"placebo_p":cnt/n_placebo,"n":len(sub)}
    else:
        z = (sub["_x"] - sub["_x"].mean()) / sub["_x"].std()
        X = sm.add_constant(pd.concat([z.rename("_x"), sub[["_prevy"]+MH_CONF]], axis=1))
        try:
            r = sm.Logit(sub["_y"], X).fit(disp=0)
        except Exception:
            return None
        return {"kind":"cont","effect":float(np.exp(r.params["_x"])),"unit":"OR/SD",
                "p":float(r.pvalues["_x"]),"placebo_p":np.nan,"n":len(sub)}

mh_rows = []
for outcome in MH_OUTCOMES:
    for cause in MH_CAUSES:
        r = mh_effect(cause, outcome, lag=1)   # prior-day cause -> today's symptom
        if r and r["p"] < 0.05:
            mh_rows.append({"outcome":outcome,"cause":cause[3:] if cause.startswith("do_") else cause,
                            **r})
mh_df = pd.DataFrame(mh_rows)

if not MH_OUTCOMES:
    print("\nNo usable mental-health outcomes (need >=60 logged days and 3-97% yes-rate).")
for outcome in MH_OUTCOMES:
    good_dir = "higher = better" if outcome in GOOD_WHEN_HIGH else "lower = better"
    print(f"\n### {outcome}  ({good_dir})")
    sub = mh_df[mh_df.outcome==outcome].sort_values("p") if len(mh_df) else mh_df
    if len(sub):
        for _, r in sub.iterrows():
            flag = "" if (r["kind"]=="cont" or r["placebo_p"]<0.05) else "  (weak: placebo n.s.)"
            if r["kind"]=="binary":
                print(f"   {r['cause']:34s} prev-day: {r['effect']:+6.1f} pp risk   p={r['p']:.4f}{flag}")
            else:
                print(f"   {r['cause']:34s} prev-day: OR={r['effect']:.2f}/SD       p={r['p']:.4f}")
    else:
        print("   (no significant prior-day causes)")


Mental-health outcomes (3): ['brain_fog', 'fatigue', 'emotionally_stable']
Candidate causes: 13 behaviors + 8 sleep/physio levers
Days with outcome data: brain_fog=121, fatigue=122, emotionally_stable=66

### brain_fog  (lower = better)
   (no significant prior-day causes)

### fatigue  (lower = better)
   used_social_media                  prev-day:  -17.6 pp risk   p=0.0001  (weak: placebo n.s.)
   took_a_cold_shower                 prev-day:  -17.9 pp risk   p=0.0232
   consumed_fruits_and_or_vegetables  prev-day:  -14.6 pp risk   p=0.0276  (weak: placebo n.s.)

### emotionally_stable  (higher = better)
   (no significant prior-day causes)


In [12]:
# Chart: prior-day behaviors -> mental-health risk (percentage-point change)
bin_mh = mh_df[mh_df.kind == "binary"].copy() if len(mh_df) else pd.DataFrame()
if len(bin_mh):
    bin_mh["label"] = bin_mh["cause"] + " → " + bin_mh["outcome"]
    bin_mh["solid"] = bin_mh["placebo_p"] < 0.05
    bin_mh = bin_mh.reindex(bin_mh.effect.abs().sort_values().index)
    # color: green = good direction, red = bad direction (depends on outcome polarity)
    def good(row):
        improves = (row.effect > 0) if row.outcome in GOOD_WHEN_HIGH else (row.effect < 0)
        return improves
    colors = [("#2d8a4e" if good(r) else "#c0392b") if r.solid else "#bbbbbb"
              for _, r in bin_mh.iterrows()]
    fig = go.Figure(go.Bar(x=bin_mh["effect"], y=bin_mh["label"], orientation="h",
                           marker_color=colors,
                           text=[f"{e:+.0f}pp" for e in bin_mh["effect"]], textposition="outside"))
    fig.add_vline(x=0, line_dash="dash", line_color="#444")
    fig.update_layout(
        title="Prior-day behavior → next-day mental state<br>"
              "<sub>green = helps · red = hurts (solid = survives placebo) · grey = weak</sub>",
        xaxis_title="change in symptom probability (percentage points)",
        height=max(420, len(bin_mh) * 30), width=900, margin=dict(l=320))
    fig.show()

# Continuous causes (sleep/HRV/bedtime) as odds ratios
cont_mh = mh_df[mh_df.kind == "cont"].copy() if len(mh_df) else pd.DataFrame()
if len(cont_mh):
    print("Sleep / physiology → mental state (odds ratio per +1 SD; OR>1 raises the flag, <1 lowers it):")
    for _, r in cont_mh.sort_values("p").iterrows():
        direction = "raises" if r.effect > 1 else "lowers"
        print(f"   {r['cause']:26s} {direction} P({r['outcome']}) — OR={r.effect:.2f}/SD (p={r.p:.3f})")


### Reading the mental-health results

- **Percentage points (pp):** "creatine → +50pp energized" means on days after taking creatine you reported feeling energized 50 points more often than days you didn't. For brain fog/fatigue, **negative is good** (less symptom).
- **Odds ratio per SD (OR/SD):** for continuous causes like sleep duration — OR<1 means more of it *lowers* the odds of the symptom.
- **Colors:** green = moves you in the good direction, red = bad direction, grey = didn't survive the placebo shuffle.

**Caveats specific to mood data (read before acting):**
- These flags are **self-reported and subjective** — "brain fog" depends on what you noticed and chose to log that day.
- **Reverse causation is rampant with mood**: feeling foggy might change what you do (e.g. you drink more caffeine *because* you're tired), so a "caffeine → energy" link can run backwards.
- Rare symptoms (logged <15% of days) have wide uncertainty even when "significant."
- Strongest, most trustworthy pattern to act on: anything where **better sleep / earlier bedtime → less fog/fatigue**, since that direction is hard to reverse-explain and matches Phase 8.
